In [1]:
import os
import json
import faiss
import numpy as np
from openai import OpenAI
import traceback
import openai  # 추가
from dotenv import load_dotenv
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import jsonlines
import MeCab
from rank_bm25 import BM25Okapi

# .env 파일 로드
load_dotenv('/upstage-ai-advanced-ir7/.env')

# API_KEY 값을 가져옴
openai_api_key = os.getenv('OPENAI_API_KEY')
upstage_api_key = os.getenv('UPSTAGE_API_KEY')



os.environ["OPENAI_API_KEY"] = openai_api_key

# Upstage API 클라이언트 설정
client = OpenAI(
    api_key= upstage_api_key,
    base_url="https://api.upstage.ai/v1/solar"
)

client_gpt = OpenAI()

In [2]:
mecab = MeCab.Tagger()

# JSONL 파일 경로
jsonl_file_path = '/upstage-ai-advanced-ir7/data/documents.jsonl'

# 문서와 docid 저장할 리스트
documents = []
docids = []

stoptags = {"E", "J", "SC", "SE", "SF", "VCN", "VCP", "VX"}

# JSONL 파일 읽기
with jsonlines.open(jsonl_file_path) as reader:
    for obj in reader:
        docids.append(obj['docid'])      # docid 저장
        documents.append(obj['content']) # content 저장

# Mecab을 사용하여 문서 토큰화
def tokenize_with_mecab(text):
    tokens = mecab.parse(text).splitlines()  # Mecab 결과를 라인 단위로 분리
    processed_tokens = []
    
    for token in tokens:
        if "\t" in token:  # 형태소와 품사 태그가 \t로 구분됨
            word, tag_info = token.split("\t")
            pos_tag = tag_info.split(",")[0]  # 품사 태그는 ,로 구분된 첫 번째 요소
            if pos_tag not in stoptags:  # 불필요한 품사 태그가 아닌 경우에만 추가
                processed_tokens.append(word)
    
    return processed_tokens

tokenized_corpus = [tokenize_with_mecab(doc) for doc in documents]

# BM25 인덱서 생성
bm25 = BM25Okapi(tokenized_corpus)

{'42508ee0-c543-4338-878e-d98c6babee66': {'docid': '42508ee0-c543-4338-878e-d98c6babee66',
  'src': 'ko_mmlu__nutrition__test',
  'content': '건강한 사람이 에너지 균형을 평형 상태로 유지하는 것은 중요합니다. 에너지 균형은 에너지 섭취와 에너지 소비의 수학적 동등성을 의미합니다. 일반적으로 건강한 사람은 1-2주의 기간 동안 에너지 균형을 달성합니다. 이 기간 동안에는 올바른 식단과 적절한 운동을 통해 에너지 섭취와 에너지 소비를 조절해야 합니다. 식단은 영양가 있는 식품을 포함하고, 적절한 칼로리를 섭취해야 합니다. 또한, 운동은 에너지 소비를 촉진시키고 근육을 강화시킵니다. 이렇게 에너지 균형을 유지하면 건강을 유지하고 비만이나 영양 실조와 같은 문제를 예방할 수 있습니다. 따라서 건강한 사람은 에너지 균형을 평형 상태로 유지하는 것이 중요하며, 이를 위해 1-2주의 기간 동안 식단과 운동을 조절해야 합니다.'},
 '4a437e7f-16c1-4c62-96b9-f173d44f4339': {'docid': '4a437e7f-16c1-4c62-96b9-f173d44f4339',
  'src': 'ko_mmlu__conceptual_physics__test',
  'content': '수소, 산소, 질소 가스의 혼합물에서 평균 속도가 가장 빠른 분자는 수소입니다. 수소 분자는 가장 가볍고 작은 원자로 구성되어 있기 때문에 다른 분자들보다 더 빠르게 움직입니다. 이러한 이유로 수소 분자는 주어진 온도에서 가장 빠른 평균 속도를 가지고 있습니다. 수소 분자는 화학 반응에서도 활발하게 참여하며, 수소 연료로도 널리 사용됩니다. 따라서 수소 분자는 주어진 온도에서 평균 속도가 가장 빠른 분자입니다.'},
 'd3c68be5-9cb1-4d6e-ba18-5f81cf89affb': {'docid': 'd3c68be5-9cb1-4d6e-ba18-5f81cf89aff

In [3]:
with open("/upstage-ai-advanced-ir7/data/eval.jsonl", "r") as f:
    eval_doc_mapping = [json.loads(line) for line in f]
    

doc_mapping = {}
with open("/upstage-ai-advanced-ir7/data/documents.jsonl", "r") as f:
    for line in f:
        doc = json.loads(line)
        doc_mapping[doc['docid']] = doc 

index = faiss.read_index("knn_index_cosine.faiss")

# gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

with open("chunk_mappings.json", "r") as f:
    chunk_doc_mapping = json.load(f)
    
model_path = 'Dongjin-kr/ko-reranker'

def exp_normalize(x):
    b = x.max()
    y = np.exp(x - b)
    return y / y.sum()
    
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to('cuda')
model.eval()

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, ou

순서
1. Query Transformation
2. Non-Science Question Detector

#### 1. Query Transformer

In [159]:
def query_transformer(messages):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    system_message = {
        "role": "system",
        "content": """
        당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. 
        사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. 
        대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.
        절대 답을 생성하지 마시오.
        """
    }

    # 사용자 메시지 준비
    dialogue_messages = [{"role": msg['role'], "content": msg['content']} for msg in messages]

    # LLM 호출을 위한 메시지 배열 생성
    full_message = [system_message] + dialogue_messages
    
    print(dialogue_messages)
    print(full_message)
    

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # LLM 모델 지정
        messages=full_message,
        temperature=0
    )

    # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    transformed_query = result.choices[0].message.content
    print(f"변환된 쿼리: {transformed_query}")
    return transformed_query

In [184]:
messages = [
    
    {"role": "user", "content": "사람이나 물체가 지구 위에서 땅속으로 꺼지거나 바깥으로 튕겨나가지 않고 가만히 서 있을 수 있잖아?"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유를 힘의 원리로 설명해줘."}
    
]

In [19]:
messages = [
    {"role": "user", "content": "기억 상실증 걸리면 너무 무섭겠다."},
    {"role": "assistant", "content": "네 맞습니다."},
    {"role": "user", "content": "원인이 뭘까."}
]

In [20]:
transformed_query = query_transformer(messages)

[{'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '원인이 뭘까.'}]
[{'role': 'system', 'content': '\n        당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. \n        사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. \n        대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.\n        절대 답을 생성하지 마시오.\n        '}, {'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '원인이 뭘까.'}]
변환된 쿼리: 기억 상실증 원인


In [12]:
transformed_query

'기억 상실증의 원인'

#### 2. Non-Science Query Detector

In [160]:
def science_query_detector(message):
    """
    LLM을 사용하여 단일 사용자의 질문을 검색에 적합한 쿼리로 변환.
    질문이 과학과 관련되어 않으면 '과학 관련 질문이 아닙니다'라고 답변.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    system_message = {
        "role": "system",
        "content": """
            당신은 일상 대화를 찾아내는 전문가 입니다
            질문이 매우 일상적인 경우에만 '과학 관련 질문이 아닙니다.'라고 답변하세요.
            예: 당신은 누구십니까? -> 과학 관련 질문이 아닙니다.

            위와 같은 질문이 아닌 경우 질문을 문서 검색에 알맞은 형태로 조금 바꾸어 내어 주세요.
            예: 착한 사마리아인에 대해 알려줘 -> 착한 사마리아인
            예: 여행은 뭐야? -> 여행의 뜻
            예: 우리는 왜 지구 위를 걸을 수 있을까? -> 지구 위를 걸을 수 있는 이유
            예: 야채 샐러드의 특징에 대해 알려줘.-> 야채 샐러드의 특징
        """
    }

    # 사용자 질문 준비
    user_message = {"role": message[0]['role'], "content": message[0]['content']}

    # LLM 호출을 위한 전체 메시지 배열 생성
    full_message = [system_message, user_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # LLM 모델 지정
        messages=full_message,
        temperature=0
    )

    # LLM이 생성한 응답을 반환
    llm_response = result.choices[0].message.content

    # 과학 관련 질문 여부 판단 및 처리
    if "과학 관련 질문이 아닙니다" in llm_response:
        print("과학 관련 질문이 아닙니다.")
        return "과학 관련 질문이 아닙니다."
    else:
        # 과학 관련 질문일 경우 변환된 쿼리 반환
        refined_query = llm_response.strip()
        print(f"변환된 쿼리: {refined_query}")
        return refined_query

In [249]:

message = [
    {"role": "user", "content": "과일 샐러드의 특징에 대해 알려줘."}
]

In [252]:
science_query_detector(message)

변환된 쿼리: 과일 샐러드의 특징


'과일 샐러드의 특징'

#### 3. Retrival KNN + Reranker

In [161]:
def normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return (embeddings / norms).astype(np.float32)  # float32로 변환

def min_max_normalize(ranked_docs):
    """
    Min-Max 정규화를 수행하는 함수
    Args:
        ranked_docs (list of tuple): [(id, score), (id, score), ...] 형태의 리스트
    Returns:
        normalized_ranked_docs (list of tuple): [(id, normalized_score), ...] 형태의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Min-Max 정규화
    min_score = np.min(scores)
    max_score = np.max(scores)
    
    # 0으로 나누는 오류를 방지 (최소값과 최대값이 같은 경우)
    if max_score == min_score:
        normalized_scores = np.ones_like(scores)
    else:
        normalized_scores = (scores - min_score) / (max_score - min_score)

    # 정규화된 점수와 id를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs


def z_score_normalize(ranked_docs):
    """
    Z-Score 정규화를 수행하는 함수
    Args:
        ranked_docs (list): (docid, score)의 리스트
    Returns:
        normalized_ranked_docs (list): Z-Score로 정규화된 (docid, normalized_score)의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Z-Score 정규화
    mean_score = np.mean(scores)
    std_dev = np.std(scores)
    
    # 표준편차가 0인 경우(모든 점수가 동일한 경우) 처리
    if std_dev == 0:
        normalized_scores = np.zeros_like(scores)
    else:
        normalized_scores = (scores - mean_score) / std_dev

    # 정규화된 점수와 docid를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs

def merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, p):
    """
    knn_retrieved_docs와 bm25_retrieved_docs에서 동일한 id를 가진 항목은 스코어를 p와 (1 - p)의 비율로 가중 합산하고,
    그렇지 않은 항목은 그대로 유지하며, 마지막에 점수대로 정렬하는 함수.
    
    Args:
        knn_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        bm25_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        p (float): knn 점수에 적용할 가중치 (0 <= p <= 1)
    
    Returns:
        merged_docs (list of tuple): [(id, combined_score), ...] 형태의 리스트, 점수 내림차순 정렬
    """
    # 두 리스트를 딕셔너리로 변환하여 빠르게 검색할 수 있게 함
    knn_dict = {doc_id: score for doc_id, score in knn_retrieved_docs}
    bm25_dict = {doc_id: score for doc_id, score in bm25_retrieved_docs}
    
    # 모든 unique id를 set으로 결합
    all_ids = set(knn_dict.keys()).union(set(bm25_dict.keys()))

    # 동일한 id가 있으면 스코어를 p와 (1 - p) 비율로 가중 합산
    merged_docs = []
    for doc_id in all_ids:
        knn_score = knn_dict.get(doc_id, 0)  # knn에 없으면 0으로 간주
        bm25_score = bm25_dict.get(doc_id, 0)  # bm25에 없으면 0으로 간주
        combined_score = p * knn_score + (1 - p) * bm25_score
        merged_docs.append((doc_id, combined_score))
    
    # 점수 내림차순으로 정렬
    merged_docs = sorted(merged_docs, key=lambda x: x[1], reverse=True)
    
    return merged_docs

def retrieval_with_score(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 500개의 유사한 chunk 검색
    k = 200
    distances, indices = index.search(normalized_query, k)


    retrieved_doc_ids = []

    # 코사인 거리를 코사인 유사도로 변환 (1 - 거리)
    for i, idx in enumerate(indices[0]):
        chunk_info = chunk_doc_mapping[idx]
        # content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        
        # 코사인 거리를 코사인 유사도로 변환
        cosine_distance = distances[0][i]
        cosine_similarity = 1 - cosine_distance  # 유사도는 1에서 거리값을 뺀 것

        # retrieved_chunks.append((query, content))
        retrieved_doc_ids.append((doc_id, cosine_similarity))
        # retrieved_scores.append(cosine_similarity)  # 유사도 점수 추가
    
    knn_retrieved_docs = z_score_normalize(retrieved_doc_ids)
    
    # BM25
    
    tokenized_query = tokenize_with_mecab(query)

    # BM25로 점수 계산
    doc_scores = bm25.get_scores(tokenized_query)

    # 점수가 높은 순서대로 문서 정렬 (상위 200개만)
    ranked_docs = sorted(zip(docids, doc_scores), key=lambda x: x[1], reverse=True)[:k]
    
    bm25_retrieved_docs = z_score_normalize(ranked_docs)

    merged_docs = merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, 0.7)
    
    retrieved_chunks = []
    for i, idx in enumerate(merged_docs):
        doc_info = doc_mapping[idx[0]]
        content = doc_info['content']
        retrieved_chunks.append((query, content))



    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)

    # 상위 5개의 문서 출력
    final_docs = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = merged_docs[i][0]
        final_docs.append((doc_id, scores[i].item()))
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)
        
    final_docs = z_score_normalize(final_docs)
    
    final_merged_docs = merge_and_sum_scores(final_docs, merged_docs, 0.475)
    
    
    return final_merged_docs[:5]
    

In [ ]:
result = retrieval_with_score('나무의 생태계 역할')

#### 3.1 LLM reranker

In [163]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    
    
    
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}: {doc_mapping[doc[0]]['content']} \n"

    # 시스템 메시지 생성
    system_message = {
        "role": "system",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서들을 찾고 그 순서를 반환하는 전문가입니다.
        
        다음의 규칙을 반드시 따르세요:
        1. 주어진 질문과 문서들의 내용을 비교하여, 질문의 답을 찾을 수 있는 문서를 가장 적합한 순서대로 나열하세요.
        2. 반드시 리스트 형태로만 출력하세요. 다른 형식은 허용되지 않습니다. 예를 들어 [1, 3, 2, 4]와 같이 반환하세요.
        3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        4. 리스트 외의 다른 정보를 출력하지 마세요. 리스트 이외의 내용은 모델이 자동으로 무시해야 합니다.

        질문: {query}

        문서들:
        {check_doc}
        """
    }

# 사용자 메시지 준비


    # 사용자 메시지 준비
    # system_message['content'] = system_message['content'] + check_doc

    # LLM 호출을 위한 메시지 배열 생성
    system_message = [system_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # LLM 모델 지정
        messages=system_message,
        temperature=0
    )

    # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    transformed_query = result.choices[0].message.content
    print(f"변환된 쿼리: {transformed_query}")
    return transformed_query

In [154]:
llm_result = llm_reranking('달의 한쪽 면만 보이는 이유', result)

변환된 쿼리: [2, 3, 1, 4, 5]


#### 4. Top Agent

In [164]:
def top_agent_without_relevance_check(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    retrieved_doc = retrieval_with_score(checked_query)
    
    check_list = []

    for doc in retrieved_doc:
        print(retrieved_doc[0][1] * 0.63)
        print(doc[1])
        print('*' * 100)
        if doc[1] > retrieved_doc[0][1] * 0.63:
            check_list.append(doc)
    
    if len(check_list) > 1:
        llm_result = llm_reranking(checked_query, check_list)
        llm_result = ast.literal_eval(llm_result)
            
        
        new_result = []
        for f_r in llm_result:
            # print(result[f_r  - 1])
            new_result.append(retrieved_doc[f_r - 1])

        retrieved_doc = new_result + retrieved_doc[len(new_result):]
    
    
    final_result = [checked_query]
    for doc in retrieved_doc:
        # current_doc = doc_mapping[doc[0]]
        # checked = document_relevance_checker(current_doc, checked_query)
        final_result.append((doc[0], doc[1]))
        
    # sorted_slice = sorted(final_result[1:], key=lambda x: x[1], reverse=True)
    # final_result[1:] = sorted_slice
    return final_result

In [149]:
let_see_2 = top_agent_without_relevance_check([{"role": "user", "content": "나무의 분류에 대해 조사해 보기 위한 방법은?"}])

변환된 쿼리: 나무의 분류 방법
상위 5개의 문서:
3.3107282560050293
5.255124215880999
****************************************************************************************************
3.3107282560050293
2.692293622697727
****************************************************************************************************
3.3107282560050293
2.676450241618026
****************************************************************************************************
3.3107282560050293
1.9075196258003642
****************************************************************************************************
3.3107282560050293
1.881075850469356
****************************************************************************************************


In [150]:
let_see_2

['나무의 분류 방법',
 ('c63b9e3a-716f-423a-9c9b-0bcaa1b9f35d', 5.255124215880999),
 ('a9f2c21e-9d44-4dd5-bc02-9e4f84077139', 2.692293622697727),
 ('8018337f-15cb-4341-b6fa-e311b4372df9', 2.676450241618026),
 ('9712bdf6-9419-4953-a8f1-8a4015dee986', 1.9075196258003642),
 ('35395c59-d1e0-4b63-803c-590220e906ad', 1.881075850469356)]

In [165]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_14_1.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

변환된 쿼리: 나무의 분류 방법
상위 5개의 문서:
3.3107282560050293
5.255124215880999
****************************************************************************************************
3.3107282560050293
2.692293622697727
****************************************************************************************************
3.3107282560050293
2.676450241618026
****************************************************************************************************
3.3107282560050293
1.9075196258003642
****************************************************************************************************
3.3107282560050293
1.881075850469356
****************************************************************************************************
****************************************************************************************************
변환된 쿼리: 각 나라의 공교육 지출 현황
상위 5개의 문서:
5.986031671427866
9.501637573695024
****************************************************************************************************
5.98603